In [1]:
import pandas as pd

# =============================
# Dataset Loading
# =============================

games_df = pd.read_csv("../Data/api data/Old data/Final Database/games.csv")
detailed_games_df = pd.read_csv("../Data/api data/Old data/Final Database/detailed_games.csv")
company_games_df = pd.read_csv("../Data/api data/Old data/Final Database/company_games.csv")
genre_games_df = pd.read_csv("../Data/api data/Old data/Final Database/genre_games.csv")
platform_df = pd.read_csv("../Data/api data/Old data/Final Database/platform.csv")


final_df = pd.read_csv("../Data/Dataset.csv")

/var/folders/wk/tl2drvdn73s_lt_qcp_4ksx40000gn/T/ipykernel_38472/1055050788.py:14: DtypeWarning: Columns (8,25) have mixed types. Specify dtype option on import or set low_memory=False.
  final_df = pd.read_csv("../Data/Dataset.csv")


In [2]:
final_df.head()
final_df.columns
print(final_df.shape)
print(final_df[['id', 'name']].head())



(159463, 81)
         id                       name
0  419296.0                     Gotcha
1   18080.0                  Half-Life
2    5159.0     Resident Evil 2 (1998)
3   19299.0                  Fallout 2
4    5426.0  Crash Bandicoot 3: Warped


In [25]:
import dash
from dash import dcc, html, Input, Output, State, ctx, ALL
import dash_bootstrap_components as dbc
import threading
import webbrowser
import socket
import pandas as pd
import squarify
import numpy as np

# =========================
# App Setup
# =========================

app = dash.Dash(
    __name__,
    external_stylesheets=[dbc.themes.BOOTSTRAP],
    suppress_callback_exceptions=True
)
app.title = "Dashboard"

def get_free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

PORT = get_free_port()

# =========================
# Component Builders
# =========================

def build_top_control(active_tab):
    return dbc.ButtonGroup(
        [
            dbc.Button(
                "Games",
                id="games-button",
                n_clicks=0,
                color="primary" if active_tab == "games" else "secondary"
            ),
            dbc.Button(
                "Trends",
                id="companies-button",
                n_clicks=0,
                color="primary" if active_tab == "trends" else "secondary"
            )
        ],
        id="top_control",
        style={"width": "100%"}
    )

def build_search_bar():
    return dbc.Card(
        dbc.CardBody(
            [
                dbc.Input(
                    id="search_bar",
                    placeholder="Search...",
                    type="text",
                    debounce=False
                ),
                html.Div(
                    id="search_results",
                    style={
                        "position": "absolute",
                        "zIndex": 1000,
                        "width": "100%",
                        "backgroundColor": "white",
                        "border": "1px solid #ced4da",
                        "borderRadius": "0.25rem",
                        "boxShadow": "0 2px 6px rgba(0,0,0,0.2)",
                        "maxHeight": "200px",
                        "overflowY": "auto",
                        "marginTop": "2px"
                    }
                )
            ]
        ),
        style={
            "position": "relative",
            "marginTop": "1rem",
            "marginBottom": "1rem",
            "backgroundColor": "white"
        }
    )

def build_games_middle(selected_sub, selected_sort_options):
    options = ["Most Popular"]
    sub_buttons = [
        dbc.Button(
            label,
            id={"type": "sub-button", "index": label.lower().replace(" ", "-") + "-sub"},
            color="primary" if selected_sub == label.lower().replace(" ", "-") + "-sub" else "secondary",
            n_clicks=0,
            style={"width": "100%", "marginBottom": "0.5rem"}
        )
        for label in options
    ]
    sort_options = ["Rating", "YouTube", "Twitch", "Added", "Metacritic"]
    sort_buttons = [
        dbc.Button(
            m,
            id={"type": "sort-button", "index": m},
            color="primary" if m in selected_sort_options else "secondary",
            style={"width": "100%", "marginBottom": "0.25rem"}
        )
        for m in sort_options
    ]
    return html.Div(
        dbc.Row(
            [
                dbc.Col(sub_buttons, width=6),
                dbc.Col([html.H6("Sort By:"), *sort_buttons], width=6)
            ]
        )
    )

def build_sidebar(active_tab, selected_sub, selected_sort_options):
    # Games controls
    games_buttons = html.Div(
        build_games_middle(selected_sub, selected_sort_options),
        style={"display": "block" if active_tab == "games" else "none"}
    )

    # Trends controls
    genre_selector = html.Div(
        [
            html.Label("Select Genre(s)", style={"fontWeight": "bold", "marginBottom": "0.5rem"}),
            dcc.Loading(
                dcc.Checklist(
                    id="genre-checklist",
                    options=[
                        {"label": g, "value": g}
                        for g in ["Action", "Adventure", "RPG", "Strategy", "Shooter"]
                    ],
                    inputStyle={"marginRight": "0.5rem", "marginLeft": "0.5rem"},
                    style={"maxHeight": "300px", "overflowY": "auto"}
                )
            )
        ],
        style={"display": "block" if active_tab == "trends" else "none"}
    )

    # Year slider
    year_slider = html.Div(
        [
            html.Label("Select Release Year Range", style={"fontWeight": "bold", "marginTop": "1rem"}),
            dcc.RangeSlider(
                id="year-range-slider",
                min=1970,
                max=2024,
                step=1,
                marks={y: str(y) for y in range(1970, 2025, 10)},
                value=[2000, 2020],
                tooltip={"placement": "bottom", "always_visible": False},
                allowCross=False,
                updatemode="mouseup"
            )
        ],
        style={
            "marginTop": "1.5rem",
            "padding": "1rem",
            "backgroundColor": "#f1f3f5",
            "borderRadius": "0.5rem",
            "boxShadow": "inset 0 1px 3px rgba(0,0,0,0.1)",
            "width": "100%"
        }
    )

    # Number slider
    num_slider = html.Div(
        [
            html.Label("Number of games (3 – 200)", style={"fontWeight": "bold", "marginTop": "1rem"}),
            dcc.Slider(
                id="num-games-slider",
                min=3,
                max=200,
                step=1,
                value=20,
                marks={i: str(i) for i in range(10, 201, 30)},
                updatemode="drag"
            )
        ],
        style={
            "marginTop": "1.5rem",
            "padding": "1rem",
            "backgroundColor": "#f1f3f5",
            "borderRadius": "0.5rem",
            "boxShadow": "inset 0 1px 3px rgba(0,0,0,0.1)",
            "width": "100%"
        }
    )

    # Compare Games button
    compare_button = dbc.Button(
        "Compare Games",
        id="compare-button",
        n_clicks=0,
        color="info",
        style={"width": "100%", "marginTop": "1rem"}
    )

    return html.Div(
        [
            build_search_bar(),
            html.Hr(),
            build_top_control(active_tab),
            html.Hr(),
            html.Div(
                id="middle_options",
                children=[games_buttons, genre_selector, year_slider, num_slider],
                style={"paddingTop": "1rem", "paddingBottom": "1rem"}
            ),
            # ← no compare_button any more
        ],
        style={"padding": "1rem"}
    )

def build_data_view():
    return html.Div(
        [
            html.Div(
                id="main_graph",
                style={
                    "flex": "1 1 auto",
                    "minHeight": 0,
                    "position": "relative",
                    "width": "100%"
                }
            ),
            html.Div(
                id="comparison_panel",
                style={
                    "flex": "0 0 auto",
                    "display": "none",
                    "overflowY": "auto"
                }
            )
        ],
        style={
            "display": "flex",
            "flexDirection": "column",
            "width": "100%",
            "height": "100%"
        }
    )

# =========================
# App Layout
# =========================

initial_active_tab       = "games"
initial_selected_sub     = "most-popular-sub"

app.layout = dbc.Container(
    [
        dcc.Store(id="active_main_tab",           data=initial_active_tab),
        dcc.Store(id="selected_sub_button",       data=initial_selected_sub),
        dcc.Store(id="selected_sort_options",     data=[]),
        dcc.Store(id="selected_year_range",       data=[2000, 2020]),
        dcc.Store(id="selected_game_ids",         data=[]),
        dcc.Store(id="last_clicked_timestamp",    data=None),

        dbc.Row(
            [
                dbc.Col(
                    id="sidebar",
                    children=build_sidebar(initial_active_tab, initial_selected_sub, []),
                    width=3,
                    style={
                        "backgroundColor": "#f8f9fa",
                        "height": "100vh",
                        "padding": 0,
                        "borderRight": "1px solid #dee2e6",
                        "display": "flex",
                        "flexDirection": "column"
                    }
                ),
                dbc.Col(
                    build_data_view(),
                    width=9,
                    style={
                        "display": "flex",
                        "flexDirection": "column",
                        "height": "100vh",
                        "padding": 0
                    }
                )
            ]
        )
    ],
    fluid=True
)

# =========================
# Callbacks
# =========================

@app.callback(
    Output("search_results", "children"),
    Input("search_bar", "value"),
    prevent_initial_call=True
)
def update_search_results(search_text):
    if not search_text or len(search_text.strip()) < 2:
        return ""
    mask = final_df["name"].str.contains(search_text, case=False, na=False)
    matches = final_df.loc[mask].head(10)
    if matches.empty:
        return html.Small("No matches", style={"color": "#888"})
    return dbc.ListGroup(
        [
            dbc.ListGroupItem(
                html.Span(
                    row["name"],
                    id={"type": "game-link", "index": int(row["id"])},
                    n_clicks=0,
                    style={
                        "color": "#0d6efd",
                        "cursor": "pointer",
                        "textDecoration": "underline"
                    }
                ),
                style={"padding": "0.4rem 0.6rem"}
            )
            for _, row in matches.iterrows()
        ],
        flush=True
    )

@app.callback(
    Output("selected_sort_options", "data"),
    Input({"type": "sort-button", "index": ALL}, "n_clicks"),
    State("selected_sort_options", "data"),
    prevent_initial_call=True
)
def select_sort_option(n_clicks_list, selected_sort_options):
    triggered = ctx.triggered_id
    if not triggered:
        return dash.no_update
    return [triggered["index"]]

@app.callback(
    Output("selected_game_ids", "data"),
    Output("last_clicked_timestamp", "data"),
    Input({"type": "game-link",   "index": ALL}, "n_clicks_timestamp"),
    Input({"type": "remove-game", "index": ALL}, "n_clicks"),
    State({"type": "game-link",   "index": ALL}, "id"),
    State("selected_game_ids",    "data"),
    State("last_clicked_timestamp","data"),
    prevent_initial_call=True
)
def handle_game_selection_and_removal(n_clicks_ts, remove_clicks, ids, selected_ids, last_ts):
    if selected_ids is None:
        selected_ids = []
    triggered = ctx.triggered_id
    # Removal
    if isinstance(triggered, dict) and triggered.get("type") == "remove-game":
        removed = triggered["index"]
        return [gid for gid in selected_ids if gid != removed], last_ts
    # Selection
    max_ts, clicked = -1, None
    for ts, btn_id in zip(n_clicks_ts, ids):
        if ts and ts > max_ts:
            max_ts, clicked = ts, btn_id["index"]
    if clicked and (last_ts is None or max_ts > last_ts):
        if clicked in selected_ids:
            return selected_ids, max_ts
        if len(selected_ids) == 2:
            selected_ids.pop(0)
        selected_ids.append(clicked)
        return selected_ids, max_ts
    return dash.no_update, dash.no_update

@app.callback(
    Output("active_main_tab", "data"),
    Input("games-button",    "n_clicks"),
    Input("companies-button","n_clicks"),
    prevent_initial_call=True
)
def update_active_tab(games_clicks, companies_clicks):
    tid = ctx.triggered_id
    if tid == "games-button":
        return "games"
    if tid == "companies-button":
        return "trends"
    return dash.no_update

@app.callback(
    Output("sidebar", "children"),
    Input("active_main_tab",           "data"),
    State("selected_sub_button",       "data"),
    State("selected_sort_options",     "data")
)
def update_sidebar(active_tab, selected_sub, selected_sort_options):
    return build_sidebar(active_tab, selected_sub, selected_sort_options)

@app.callback(
    Output("genre-checklist", "options"),
    Input("active_main_tab", "data")
)
def populate_genres(tab):
    if tab != "trends":
        return []
    import ast
    genres = []
    for g in final_df["genres"].dropna():
        try:
            parsed = ast.literal_eval(g) if isinstance(g, str) else g
            if isinstance(parsed, list):
                genres.extend(parsed)
        except:
            continue
    uniq = sorted(set(genres))
    return [{"label": g, "value": g} for g in uniq]

@app.callback(
    Output("main_graph", "children"),
    Input("active_main_tab",       "data"),
    Input("selected_sort_options","data"),
    Input("selected_year_range",   "data"),
    Input("num-games-slider",      "value"),
    Input("genre-checklist",       "value"),
    prevent_initial_call=True
)
def update_main_graph(active_tab, selected_sort_options, selected_year_range, num_games, selected_genres):
    df = final_df.copy()
    df = df.dropna(subset=["name"]).drop_duplicates()
    df["release_year"] = pd.to_datetime(df["released"], errors="coerce").dt.year
    df = df[
        (df["release_year"] >= selected_year_range[0]) &
        (df["release_year"] <= selected_year_range[1])
    ]

    if active_tab == "trends":
        import ast, statsmodels.api as sm, plotly.graph_objects as go
        fdf = df.dropna(subset=["released", "rating", "genres"])
        fdf["released"] = pd.to_datetime(fdf["released"], errors="coerce")
        fdf = fdf[fdf["rating"] > 0.5]
        fdf = fdf[
            (fdf["released"].dt.year >= selected_year_range[0]) &
            (fdf["released"].dt.year <= selected_year_range[1])
        ]
        def extract_matching_genre(s):
            try:
                lst = ast.literal_eval(s)
                for g in lst:
                    if g in selected_genres:
                        return g
            except:
                pass
            return None
        if selected_genres:
            fdf["matched_genre"] = fdf["genres"].apply(extract_matching_genre)
            fdf = fdf[fdf["matched_genre"].notna()]
            color_col = "matched_genre"
        else:
            fdf["matched_genre"] = "All"
            color_col = None
        fdf = fdf.sample(n=min(num_games, len(fdf)), random_state=42).sort_values("released")
        x = fdf["released"].map(pd.Timestamp.toordinal)
        y = fdf["rating"]
        X = sm.add_constant(x)
        model = sm.OLS(y, X).fit()
        trend_y = model.predict(X)
        fig = go.Figure()
        fig.add_trace(
            go.Scatter(
                x=fdf["released"],
                y=fdf["rating"],
                mode="markers",
                marker=dict(size=4, opacity=0.4),
                text=fdf["name"],
                marker_color=(fdf[color_col] if color_col else None),
                showlegend=bool(color_col)
            )
        )
        fig.add_trace(
            go.Scatter(
                x=fdf["released"],
                y=trend_y,
                mode="lines",
                line=dict(color="orange", width=2),
                name="Trendline"
            )
        )
        fig.update_layout(
            title="Game Ratings Over Time" + (" by Genre" if selected_genres else ""),
            xaxis_title="Release Date",
            yaxis_title="Rating",
            yaxis_range=[0.5, 5],
            margin=dict(l=20, r=20, t=50, b=20)
        )
        return dcc.Graph(figure=fig, style={"height": "100%", "width": "100%"})

    # Heatmap fallback
    col_map = {
        "rating": "rating",
        "youtube": "youtube_count",
        "twitch": "twitch_count",
        "added": "added",
        "metacritic": "metacritic"
    }
    sort_by = col_map.get(
        (selected_sort_options[0] if selected_sort_options else "rating").lower(),
        "rating"
    )
    df2 = (
        df.dropna(subset=[sort_by])
          .sort_values(sort_by, ascending=False)
          .head(num_games or 50)
    )
    areas  = df2[sort_by].astype(float).clip(lower=1e-6)
    normed = squarify.normalize_sizes(areas, 100, 100)
    rects  = squarify.squarify(normed, 0, 0, 100, 100)
    df2 = pd.concat([df2.reset_index(drop=True), pd.DataFrame(rects)], axis=1)

    tiles = []
    for _, row in df2.iterrows():
        tiles.append(
            html.Div(
                [
                    html.Div(
                        row["name"],
                        style={
                            "fontSize": "12px",
                            "fontWeight": "bold",
                            "overflow": "hidden",
                            "textOverflow": "ellipsis",
                            "whiteSpace": "nowrap"
                        }
                    ),
                    html.Div(f"{sort_by.capitalize()}: {row[sort_by]:.2f}", style={"fontSize": "10px"})
                ],
                style={
                    "position": "absolute",
                    "left":   f"{row['x']:.2f}%",
                    "top":    f"{row['y']:.2f}%",
                    "width":  f"{row['dx']:.2f}%",
                    "height": f"{row['dy']:.2f}%",
                    "backgroundImage":  f"url('{row['background_image']}')",
                    "backgroundSize":   "cover",
                    "backgroundPosition": "center",
                    "border": "1px solid #fff",
                    "boxSizing": "border-box",
                    "borderRadius": "4px",
                    "overflow": "hidden",
                    "color": "#fff"
                }
            )
        )

    return html.Div(
        tiles,
        style={
            "position": "relative",
            "width": "100%",
            "height": "100%",
            "backgroundColor": "#333"
        }
    )

@app.callback(
    Output("comparison_panel", "children"),
    Output("comparison_panel", "style"),
    Input("selected_game_ids", "data"),
)
def update_comparison_panel(game_ids):
    # Hide if no games selected
    if not game_ids:
        return [], {"display": "none"}

    # Determine how many columns: up to 3 per row
    slots = min(len(game_ids), 3)
    col_width = 12 // slots   # 1→12, 2→6, 3→4

    cols = []
    for game_id in game_ids:
        match = final_df[final_df["id"] == game_id]
        if match.empty:
            continue
        row = match.iloc[0]

        # Build the card content
        card = dbc.Card(
            [
                dbc.CardHeader(
                    html.Div(
                        [
                            html.Span(
                                row.get("name", "Unnamed Game"),
                                style={"fontWeight": "bold", "fontSize": "1.2rem"}
                            ),
                            html.Button(
                                "×",
                                id={"type": "remove-game", "index": game_id},
                                style={
                                    "float": "right",
                                    "border": "none",
                                    "background": "none",
                                    "fontSize": "20px",
                                    "cursor": "pointer",
                                    "color": "red"
                                }
                            )
                        ]
                    )
                ),
                html.Div(
                    [
                        html.P(f"Rating: {row.get('rating', 'N/A')}"),
                        html.P(f"Metacritic: {row.get('metacritic', 'N/A')}"),
                        html.P(f"Released: {row.get('released', 'Unknown')}"),
                        html.P(f"Added: {row.get('added', 'N/A')}"),
                        html.P(f"YouTube: {row.get('youtube_count', 'N/A')}"),
                        html.P(f"Twitch: {row.get('twitch_count', 'N/A')}"),
                        html.P(
                            (row.get("description_raw") or "")[:180] + "..."
                        ) if isinstance(row.get("description_raw"), str)
                          else html.P("No description available.")
                    ],
                    style={
                        "position": "relative",
                        "backgroundImage": f"url('{row.get('background_image', '')}')",
                        "backgroundSize": "cover",
                        "backgroundPosition": "center",
                        "borderRadius": "0 0 8px 8px",
                        "padding": "0.75rem",
                        "color": "white",
                        "flex": "1 1 auto",
                        "overflow": "auto"
                    }
                )
            ],
            style={"display": "flex", "flexDirection": "column", "height": "100%"}
        )

        # Wrap in a Col of dynamic width
        cols.append(dbc.Col(card, width=col_width, style={"padding": "0.5rem", "height": "300px"}))

    # Use a Row so Bootstrap grid will place up to 3 per line,
    # and additional cards will wrap automatically.
    row = dbc.Row(cols, justify="start", style={"width": "100%"})

    return row, {"display": "block", "width": "100%"}





@app.callback(
    Output("selected_year_range", "data"),
    Input("year-range-slider", "value"),
    prevent_initial_call=True
)
def update_selected_year_range(year_range):
    return year_range


# =========================
# Run server
# =========================

def open_browser():
    webbrowser.open_new(f"http://127.0.0.1:{PORT}")

if __name__ == "__main__":
    threading.Timer(1, open_browser).start()
    app.run(debug=True, use_reloader=False, port=PORT)


---------------------------------------------------------------------------
error                                     Traceback (most recent call last)
Cell In[25], line 310, in update_search_results(search_text='God of War (')
    308 if not search_text or len(search_text.strip()) < 2:
    309     return ""
--> 310 mask = final_df["name"].str.contains(search_text, case=False, na=False)
        search_text = 'God of War ('
        final_df =               id                                        slug  \
0       419296.0                                    gotcha-2   
1        18080.0                                   half-life   
2         5159.0                             resident-evil-2   
3        19299.0  fallout-2-a-post-nuclear-role-playing-game   
4         5426.0                      crash-bandicoot-warped   
...          ...                                         ...   
159458  997390.0              super-bomberman-panic-bomber-w   
159459  973008.0           ginga-yukyou-de

/var/folders/wk/tl2drvdn73s_lt_qcp_4ksx40000gn/T/ipykernel_38472/3919167352.py:310: UserWarning:

This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.



---------------------------------------------------------------------------
error                                     Traceback (most recent call last)
Cell In[25], line 310, in update_search_results(search_text='God of War (2018')
    308 if not search_text or len(search_text.strip()) < 2:
    309     return ""
--> 310 mask = final_df["name"].str.contains(search_text, case=False, na=False)
        search_text = 'God of War (2018'
        final_df =               id                                        slug  \
0       419296.0                                    gotcha-2   
1        18080.0                                   half-life   
2         5159.0                             resident-evil-2   
3        19299.0  fallout-2-a-post-nuclear-role-playing-game   
4         5426.0                      crash-bandicoot-warped   
...          ...                                         ...   
159458  997390.0              super-bomberman-panic-bomber-w   
159459  973008.0           ginga-y

/var/folders/wk/tl2drvdn73s_lt_qcp_4ksx40000gn/T/ipykernel_38472/3919167352.py:441: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

